In [53]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Note: you may need to restart the kernel to use updated packages.


In [54]:
import os

print("Project ID found:", bool(os.environ.get("gcl_project_id")))
print("GCP service account key found:", bool(os.environ.get("GCP_SA_KEY")))

Project ID found: True
GCP service account key found: True


In [55]:
import sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ["gcl_project_id"]   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: datalab-504011


In [56]:
# Завдання 6. Ноутбук 05 — історична таблиця фактів і злиття

# Завдання 6.1. Прочитайте з nbu_raw.raw_rates рядки тільки за сьогодні.

query = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
WHERE business_date = CURRENT_DATE()
"""

# Завдання 6.2. Розгорніть payload, нормалізуйте значення й приберіть дублікати усередині свого набору

raw_rates_bq = client.query(query).to_dataframe()
raw_rates_bq

payload = pd.json_normalize(raw_rates_bq["payload"].map(json.loads))

payload["ingested_at"] = raw_rates_bq["ingested_at"].values
payload["business_date"] = raw_rates_bq["business_date"].values

payload["cc"] = payload["cc"].str.strip().str.upper()
payload["txt"] = payload["txt"].str.strip()
payload["r030"] = payload["r030"].astype("Int64")

payload = payload.sort_values("ingested_at").drop_duplicates(subset=["business_date", "cc"], keep="last").reset_index(drop=True)


payload



/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cc,exchangedate,r030,rate,special,txt,ingested_at,business_date
0,DZD,31.08.2026,12,0.334950,None,Алжирський динар,2026-08-29 11:56:16.108246,2026-08-29
1,AUD,31.08.2026,36,32.049200,None,Австралійський долар,2026-08-29 11:56:16.108263,2026-08-29
2,BDT,31.08.2026,50,0.363330,None,Така,2026-08-29 11:56:16.108275,2026-08-29
3,CAD,31.08.2026,124,32.166200,None,Канадський долар,2026-08-29 11:56:16.108286,2026-08-29
4,CNY,31.08.2026,156,6.628100,None,Юань Женьміньбі,2026-08-29 11:56:16.108350,2026-08-29
5,CZK,31.08.2026,203,2.148200,None,Чеська крона,2026-08-29 11:56:16.108371,2026-08-29
6,DKK,31.08.2026,208,6.940800,None,Данська крона,2026-08-29 11:56:16.108382,2026-08-29
7,HKD,31.08.2026,344,5.682700,None,Гонконгівський долар,2026-08-29 11:56:16.108394,2026-08-29
8,HUF,31.08.2026,348,0.142161,None,Форинт,2026-08-29 11:56:16.108405,2026-08-29
9,INR,31.08.2026,356,0.467030,None,Індійська рупія,2026-08-29 11:56:16.108415,2026-08-29


In [57]:
# Завдання 6.2. Розгорніть payload, нормалізуйте значення й приберіть дублікати усередині свого набору
payload["ingested_at"] = raw_rates_bq["ingested_at"]
payload["business_date"] = raw_rates_bq["business_date"]

payload["cc"] = payload["cc"].str.strip().str.upper()
payload["txt"] = payload["txt"].str.strip()
payload["r030"] = payload["r030"].astype("Int64")

payload = payload.sort_values("ingested_at").drop_duplicates(subset=["business_date", "cc"], keep="last").reset_index(drop=True)


payload


,cc,exchangedate,r030,rate,special,txt,ingested_at,business_date
0,DZD,31.08.2026,12,0.334950,None,Алжирський динар,2026-08-29 11:56:16.108246+00:00,2026-08-29
1,AUD,31.08.2026,36,32.049200,None,Австралійський долар,2026-08-29 11:56:16.108263+00:00,2026-08-29
2,BDT,31.08.2026,50,0.363330,None,Така,2026-08-29 11:56:16.108275+00:00,2026-08-29
3,CAD,31.08.2026,124,32.166200,None,Канадський долар,2026-08-29 11:56:16.108286+00:00,2026-08-29
4,CNY,31.08.2026,156,6.628100,None,Юань Женьміньбі,2026-08-29 11:56:16.108350+00:00,2026-08-29
5,CZK,31.08.2026,203,2.148200,None,Чеська крона,2026-08-29 11:56:16.108371+00:00,2026-08-29
6,DKK,31.08.2026,208,6.940800,None,Данська крона,2026-08-29 11:56:16.108382+00:00,2026-08-29
7,HKD,31.08.2026,344,5.682700,None,Гонконгівський долар,2026-08-29 11:56:16.108394+00:00,2026-08-29
8,HUF,31.08.2026,348,0.142161,None,Форинт,2026-08-29 11:56:16.108405+00:00,2026-08-29
9,INR,31.08.2026,356,0.467030,None,Індійська рупія,2026-08-29 11:56:16.108415+00:00,2026-08-29


In [58]:
# Завдання 6.3. Підставте ключі вимірів: date_key — з business_date у форматі YYYYMMDD, currency_key — приєднанням nbu_dwh.dim_currency, з -1 для ненайдених валют.


date_dim_bq = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_dwh.dim_date`
"""

currency_dim_bq = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_dwh.dim_currency`
"""



date_dim = client.query(date_dim_bq).to_dataframe()

payload = payload.merge(
    date_dim[["date_key", "full_date"]],
    left_on="business_date",
    right_on="full_date",
    how="left"
)


currency_dim = client.query(currency_dim_bq).to_dataframe()

payload = payload.merge(
    currency_dim[["currency_key", "currency_code"]],
    left_on="cc",
    right_on="currency_code",
    how="left"
)

payload["dw_load_ts"] = pd.Timestamp.now(tz="UTC")

payload["currency_key"] = payload["currency_key"].fillna(-1).astype("Int64")
        
fact = payload[[
    "date_key", "currency_key", "rate", "business_date", "dw_load_ts"
]]

fact

/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_key,currency_key,rate,business_date,dw_load_ts
0,20260829,1,0.334950,2026-08-29,2026-08-29 19:04:31.199366+00:00
1,20260829,2,32.049200,2026-08-29,2026-08-29 19:04:31.199366+00:00
2,20260829,3,0.363330,2026-08-29,2026-08-29 19:04:31.199366+00:00
3,20260829,4,32.166200,2026-08-29,2026-08-29 19:04:31.199366+00:00
4,20260829,5,6.628100,2026-08-29,2026-08-29 19:04:31.199366+00:00
5,20260829,6,2.148200,2026-08-29,2026-08-29 19:04:31.199366+00:00
6,20260829,7,6.940800,2026-08-29,2026-08-29 19:04:31.199366+00:00
7,20260829,8,5.682700,2026-08-29,2026-08-29 19:04:31.199366+00:00
8,20260829,9,0.142161,2026-08-29,2026-08-29 19:04:31.199366+00:00
9,20260829,10,0.467030,2026-08-29,2026-08-29 19:04:31.199366+00:00


In [59]:
# Завдання 6.4. Створіть таблицю nbu_dwh.fact_exchange_rate, якщо її ще немає: схема з пункту 6.3, партиціювання за business_date, кластеризація за currency_key.

fact_id = f"{PROJECT_ID}.nbu_dwh.fact_exchange_rate"

schema = [
    bigquery.SchemaField("date_key", "INT64"),
    bigquery.SchemaField("currency_key", "INT64"),
    bigquery.SchemaField("rate", "FLOAT64"),
    bigquery.SchemaField("business_date", "DATE"),
    bigquery.SchemaField("dw_load_ts", "TIMESTAMP")
]

table = bigquery.Table(fact_id, schema=schema)
table.time_partitioning = bigquery.TimePartitioning(field="business_date")
table.clustering_fields = ["currency_key"]
client.create_table(table, exists_ok=True)

print("Table fact_exchange_rate created!")

Table fact_exchange_rate created!


In [60]:
# Завдання 6.5. Реалізуйте злиття: рядки за сьогодні мають бути оновлені або додані, рядки за попередні дні — залишитися недоторканими.


# 1) уся історія, КРІМ сьогодні
old = client.query(f"""
    SELECT date_key, currency_key, rate, business_date, dw_load_ts
    FROM `{fact_id}`
    WHERE business_date <> CURRENT_DATE()
""").to_dataframe()

# 2) склеюємо з новою порцією
merged = pd.concat([old, fact], ignore_index=True)

# 3) перезаписуємо таблицю цілком
cfg = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(merged, fact_id, job_config=cfg).result()


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=datalab-504011, location=EU, id=32b08205-209f-44f3-8c23-97c5a6a0edce>

In [61]:
# Завдання 6.6. Перевірте й виведіть результат трьома запитами:

# Checking for duplicates

query = f"""
SELECT date_key, currency_key, COUNT(*) AS cnt
FROM `{fact_id}`
GROUP BY date_key, currency_key
HAVING COUNT(*) > 1
"""

duplicates = client.query(query).to_dataframe()

print("Grain is unique:", len(duplicates) == 0)


# кількість різних business_date у таблиці;
query_business_day_duplicates = f"""
SELECT COUNT(DISTINCT business_date) AS date_count
FROM `{fact_id}`                 
"""

result_days = client.query(query_business_day_duplicates).to_dataframe()

print("Different business dates:", result_days["date_count"].iloc[0])


# загальна кількість рядків.

number_of_rows = f"""
SELECT COUNT(*) cnt
FROM `{fact_id}`
"""

num_rows = client.query(number_of_rows).to_dataframe()
print("Total rows:", num_rows["cnt"].iloc[0])

Grain is unique: True
Different business dates: 1
Total rows: 45


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
